# 화재 이미지 탐지 — Kaggle GPU 학습

**설정 방법:**
1. 우측 Accelerator → **GPU T4 x2** 선택
2. 데이터셋 추가 (2개 모두):
   - `+ Add Data` → `yuntarwon/fireimage-detection` (abnormal 2,871장)
   - `+ Add Data` → `yuntarwon/fireimage-normal` (normal 6,106장)
3. 셀 순서대로 실행

In [ ]:
# Cell 1: 코드 내장 zip 추출 (Windows 백슬래시 경로 처리)
import os, zipfile, base64, io, shutil

CODE_B64 = "..."

dst = '/kaggle/working/fireimage_detection'
os.makedirs(dst, exist_ok=True)

zip_bytes = base64.b64decode(CODE_B64)
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for member in z.namelist():
        # Windows 백슬래시 → Linux 슬래시 변환
        rel_path = member.replace('\\', '/').replace('\\\\', '/')
        target = os.path.join(dst, rel_path)
        if rel_path.endswith('/'):
            os.makedirs(target, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(member) as src_f, open(target, 'wb') as dst_f:
                shutil.copyfileobj(src_f, dst_f)

os.chdir(dst)
print('작업 디렉토리:', os.getcwd())
# 디렉토리 구조 확인
dirs = [d for d in os.listdir('.') if os.path.isdir(d)]
print('폴더:', dirs)


In [ ]:
# Cell 2: GPU 호환 PyTorch 설치
import subprocess, sys

# nvidia-smi로 compute capability 확인 (torch import 전)
r = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                   capture_output=True, text=True)
cap = None
if r.returncode == 0 and r.stdout.strip():
    try:
        cap = float(r.stdout.strip().split('\n')[0])
    except Exception:
        cap = None
print(f"GPU compute capability: {cap}")

# Kaggle 기본 torch는 sm_60(P100) 미지원 → 공식 빌드(sm_60 포함)로 교체
if cap is not None and cap < 7.0:
    print(f"P100(sm_{int(cap*10)}) 감지 → sm_60 지원 공식 PyTorch 설치 중... (수 분 소요)")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch==2.4.1', 'torchvision==0.19.1',
                    '--index-url', 'https://download.pytorch.org/whl/cu121',
                    '--force-reinstall', '--no-deps'], check=False)
    # 검증
    v = subprocess.run([sys.executable, '-c',
                        'import torch; print(torch.__version__, torch.cuda.get_arch_list())'],
                       capture_output=True, text=True)
    print("재설치 후:", v.stdout.strip(), v.stderr.strip()[:200])
else:
    print(f"sm_{int((cap or 7.5)*10)} 호환 OK — 기본 PyTorch 사용")

# 기타 필수 패키지
for pkg in ['timm', 'einops']:
    try:
        __import__(pkg.replace('-','_'))
        print(f'{pkg}: 이미 설치됨')
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)
        print(f'{pkg}: 설치 시도')

print("패키지 준비 완료")

In [ ]:
# Cell 3: 데이터셋 연결
import os, zipfile, glob, shutil

base = '/kaggle/working/fireimage_detection/data/fireimage'
os.makedirs(base, exist_ok=True)

print("마운트된 데이터셋:")
for d in sorted(os.listdir('/kaggle/input')):
    cnt = sum(1 for _ in glob.glob(f'/kaggle/input/{d}/**/*', recursive=True) if os.path.isfile(_))
    print(f"  {d}: {cnt}개")

# abnormal 심링크
ab_src = '/kaggle/input/fireimage-abnormal'
ab_dst = f'{base}/abnormal'
assert os.path.exists(ab_src), "fireimage-abnormal 없음"

yt_src = '/kaggle/input/fireimage-abnormal-youtube'
yt_zip = f'{yt_src}/abnormal_youtube.zip'
has_youtube = os.path.exists(yt_src) and bool(os.listdir(yt_src) if os.path.exists(yt_src) else [])

if has_youtube:
    os.makedirs(ab_dst, exist_ok=True)
    for sub in os.listdir(ab_src):
        s = os.path.join(ab_src, sub)
        d2 = os.path.join(ab_dst, sub)
        if os.path.isdir(s) and not os.path.exists(d2):
            os.symlink(s, d2)
    yt_dst = f'{ab_dst}/youtube'
    if os.path.exists(yt_zip) and not os.path.exists(yt_dst):
        os.makedirs(yt_dst, exist_ok=True)
        with zipfile.ZipFile(yt_zip) as z:
            z.extractall(yt_dst)
        print(f"youtube 추출: {len(os.listdir(yt_dst))}장")
    elif not os.path.exists(yt_dst):
        os.symlink(yt_src, yt_dst)
        print("youtube 심링크")
    print("abnormal + youtube 완료")
else:
    if not os.path.exists(ab_dst):
        os.symlink(ab_src, ab_dst)
    print("abnormal 심링크 (youtube 없음)")

# normal 심링크
nm_src = '/kaggle/input/fireimage-normal'
nm_dst = f'{base}/normal'
assert os.path.exists(nm_src), "fireimage-normal 없음"
if not os.path.exists(nm_dst):
    os.symlink(nm_src, nm_dst)
print(f"normal 심링크")

n = sum(1 for f in glob.glob(f'{base}/normal/**/*',   recursive=True) if os.path.isfile(f))
a = sum(1 for f in glob.glob(f'{base}/abnormal/**/*', recursive=True) if os.path.isfile(f))
print(f"\nnormal: {n:,}장 / abnormal: {a:,}장")
assert n > 0 and a > 0, f"데이터 없음"
print("데이터 준비 완료!")

In [ ]:
# Cell 4: GPU + 데이터 확인
import torch, glob, os

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    name = torch.cuda.get_device_name(0)
    if cap[0] >= 7:
        print(f"GPU: {name} (sm_{cap[0]}{cap[1]}) — 호환 OK")
        device_info = f"cuda ({name})"
    else:
        print(f"WARNING: {name} (sm_{cap[0]}{cap[1]}) PyTorch 호환 불가 (sm_70 이상 필요) → CPU 사용")
        device_info = f"cpu (GPU sm_{cap[0]}{cap[1]} 호환불가)"
else:
    print("GPU 없음 → CPU 사용")
    device_info = "cpu"

WD = '/kaggle/working/fireimage_detection'
os.makedirs(WD, exist_ok=True)
with open(f'{WD}/device_status.txt', 'w') as f:
    f.write(device_info)
print(f"Device: {device_info}")

base = '/kaggle/working/fireimage_detection/data/fireimage'
normal_cnt   = sum(1 for _ in glob.glob(f'{base}/normal/**/*',   recursive=True) if os.path.isfile(_))
abnormal_cnt = sum(1 for _ in glob.glob(f'{base}/abnormal/**/*', recursive=True) if os.path.isfile(_))
print(f'normal: {normal_cnt:,}장 / abnormal: {abnormal_cnt:,}장')

In [ ]:
# Cell 5: 학습 실행
import subprocess, sys, os

WD = '/kaggle/working/fireimage_detection'

print("=== main_v2.py 시작 ===")
print("현재 디렉토리:", WD)
print("data 확인:", os.listdir(f'{WD}/data/fireimage') if os.path.exists(f'{WD}/data/fireimage') else "없음")

result = subprocess.run(
    [sys.executable, 'main_v2.py', '--class_name', 'fireimage'],
    capture_output=False,
    text=True,
    cwd=WD
)
print(f"\n=== 종료 코드: {result.returncode} ===")
if result.returncode != 0:
    raise RuntimeError(f"main_v2.py 실패 (exit code {result.returncode})")
print("학습 완료!")

In [ ]:
# Cell 6: 결과 확인
import os, subprocess, sys

subprocess.run([sys.executable, 'compare_models.py'], capture_output=False)

import pandas as pd
metrics_path = 'results/fireimage/metrics.csv'
if os.path.exists(metrics_path):
    df = pd.read_csv(metrics_path)
    print(df.to_string())
else:
    print(f"metrics.csv 없음. results 폴더 내용:")
    for root, dirs, files in os.walk('results'):
        for f in files:
            print(os.path.join(root, f))

In [ ]:
# Cell 6: 결과 확인 및 비교표 생성
!python compare_models.py

import pandas as pd
df = pd.read_csv('results/fireimage/metrics.csv')
print(df.to_string())

In [ ]:
# Cell 7: model_save 압축 (세션 종료 전 반드시 실행 → Output에서 다운로드)
import shutil
shutil.make_archive('/kaggle/working/model_save_backup', 'zip', '/kaggle/working/fireimage_detection', 'model_save')
shutil.make_archive('/kaggle/working/results_backup', 'zip', '/kaggle/working/fireimage_detection', 'results')
print('압축 완료. Output 탭에서 다운로드하세요.')